In [ ]:
# ==============================
# IMPORTACIONES
# ==============================
import os
import re
import pandas as pd
import pdfplumber
from pathlib import Path

# ==============================
# CONFIGURACIÓN DE CARPETAS
# ==============================
PDF_FOLDER = "pdfs"
OUTPUT_FOLDER = "output"

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_FOLDER}/datos_extraidos.csv"

# ==============================
# FUNCIONES DE EXTRACCIÓN
# ==============================

def extraer_texto_pdf(ruta_pdf):
    texto = ""
    try:
        with pdfplumber.open(ruta_pdf) as pdf:
            for pagina in pdf.pages:
                contenido = pagina.extract_text()
                if contenido:
                    texto += contenido + "\n"
    except Exception as e:
        print(f"Error leyendo {ruta_pdf}: {e}")
    return texto


def extraer_datos(texto):
    datos = {}

    # Nombre (después de "NOMBRE")
    nombre = re.search(r"NOMBRE\s+([A-Z\s]+)", texto)
    datos["nombre"] = nombre.group(1).strip() if nombre else None

    # RFC
    rfc = re.search(r"\b[A-Z]{4}\d{6}[A-Z0-9]{3}\b", texto)
    datos["rfc"] = rfc.group() if rfc else None

    # Teléfono
    telefono = re.search(r"\b\d{10}\b", texto)
    datos["telefono"] = telefono.group() if telefono else None

    # Email
    email = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", texto)
    datos["email"] = email.group() if email else None

    # Código postal
    cp = re.search(r"C\.P\.\s*(\d{5})", texto)
    datos["cp"] = cp.group(1) if cp else None

    # Ciudad
    ciudad = re.search(r"\b([A-Z\s]+)\s+FECHA CITA", texto)
    datos["ciudad"] = ciudad.group(1).strip() if ciudad else None

    # Prospecto
    prospecto = re.search(r"NO\.\s*PROSPECTO\s*(\d+)", texto)
    datos["no_prospecto"] = prospecto.group(1) if prospecto else None

    return datos


# ==============================
# PROCESAMIENTO PRINCIPAL
# ==============================

def procesar_pdfs():
    resultados = []

    archivos_pdf = list(Path(PDF_FOLDER).glob("*.pdf"))

    print(f"Se encontraron {len(archivos_pdf)} archivos PDF\n")

    for pdf_file in archivos_pdf:
        print(f"Procesando: {pdf_file.name}")

        texto = extraer_texto_pdf(pdf_file)

        if not texto:
            print("⚠️ No se pudo extraer texto\n")
            continue

        datos = extraer_datos(texto)
        datos["archivo"] = pdf_file.name

        resultados.append(datos)

    return resultados


# ==============================
# EJECUCIÓN
# ==============================

datos = procesar_pdfs()

df = pd.DataFrame(datos)

# Limpieza básica
df = df.drop_duplicates()

# Guardar CSV
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("\n✅ Proceso finalizado")
print(f"Archivo guardado en: {OUTPUT_FILE}")

# Mostrar resultados
df.head()